# Tile staging

In [ ]:
import sys
sys.path.append('..')

from multiview_stitcher import spatial_image_utils as si_utils
import napari

from muvis_align.MVSRegistration import MVSRegistration
from muvis_align.image.util import get_sim_physical_size, create_image_shapes, extract_sims_from_fused, \
    sims_from_sims_or_msims

## Initialise muvis-align, initialise sims

In [ ]:
reg = MVSRegistration(operation='register', input_path='../data/S000/*.zarr',
                      output_path='../../output/', ui='mpl', debug=True)
reg.init_data()
msims = reg.msims
sims = sims_from_sims_or_msims(msims)

def print_dict(d):
	return ', '.join([f'{k}:<font color="blue">{v:.3f}</font>' for k, v in d.items()])


for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Calculate tile shapes

In [ ]:
shapes = create_image_shapes(msims, transform_key=reg.source_transform_key)
for label, shape in zip(reg.file_labels, shapes):
	print(label, shape[0])

## Calculate fused image

In [ ]:
fused_msim, _ = reg.fuse(msims, transform_key=reg.source_transform_key, fusion_method='additive')
fused_sim = extract_sims_from_fused(fused_msim)
fused_scale = si_utils.get_spacing_from_sim(fused_sim, asarray=True)
fused_position = si_utils.get_origin_from_sim(fused_sim, asarray=True)

## Open napari viewer

In [ ]:
viewer = napari.Viewer()

## Visualise tiles and fused image

In [ ]:
viewer.add_image(fused_sim, name='fused', scale=fused_scale, translate=fused_position)
text = {'string': '{labels}', 'color': 'blue'}
features = {'labels': reg.file_labels}
viewer.add_shapes(shapes, name=f'tile shapes', text=text, features=features, face_color='transparent')